<h2>Setup paths + imports</h3>

In [6]:
import sys
from pathlib import Path

# project root = parent of notebooks/
PROJECT_ROOT = Path.cwd().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

print("Added to PYTHONPATH:", PROJECT_ROOT)


Added to PYTHONPATH: /Users/aidos/ML Projects Personal/Comp_BioChem_Project


In [9]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

from src.baselines.heuristics import (
    BaselineConfig,
    load_chain_graph,
    baseline_surface_filtered_distance_score,
    scores_to_topk_predictions,
)

ROOT = PROJECT_ROOT
META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"

print("Metadata dir:", META)
print("Processed dir:", PROCESSED)


Metadata dir: /Users/aidos/ML Projects Personal/Comp_BioChem_Project/data/metadata
Processed dir: /Users/aidos/ML Projects Personal/Comp_BioChem_Project/data/processed


<h2> Load dataset splits

In [10]:
with open(META / "data_splits.json", "r") as f:
    splits = json.load(f)

print(
    "Train:", len(splits["train"]),
    "Val:", len(splits["val"]),
    "Test:", len(splits["test"])
)


Train: 180 Val: 35 Test: 35


<h2> Metric helpers</h2>

In [11]:
def precision_recall_auc(y_true, y_score):
    order = np.argsort(-y_score)
    y_true = y_true[order]

    tp, fp = 0, 0
    total_pos = int(y_true.sum())
    if total_pos == 0:
        return float("nan")

    pr_auc = 0.0
    prev_recall = 0.0

    for i in range(len(y_true)):
        if y_true[i] == 1:
            tp += 1
        else:
            fp += 1

        precision = tp / max(1, tp + fp)
        recall = tp / total_pos
        pr_auc += precision * (recall - prev_recall)
        prev_recall = recall

    return float(pr_auc)


def f1_at_k(y_true, y_score, k):
    pred = scores_to_topk_predictions(y_score, k)
    tp = ((pred == 1) & (y_true == 1)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    fn = ((pred == 0) & (y_true == 1)).sum()

    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    if prec + rec == 0:
        return 0.0
    return float(2 * prec * rec / (prec + rec))


def precision_at_k(y_true, y_score, k):
    pred = scores_to_topk_predictions(y_score, k)
    tp = ((pred == 1) & (y_true == 1)).sum()
    return float(tp / max(1, k))


<h2> Run Baseline 0 on a split</h2>

In [12]:
cfg = BaselineConfig(surface_density_quantile=0.40)

def eval_split(split_name, k_list=(10, 20, 30)):
    rows = []

    for ex in tqdm(splits[split_name], desc=f"Baseline0 {split_name}"):
        pdb_id, chainA, chainB = ex["pdb_id"], ex["chainA"], ex["chainB"]
        ex_dir = PROCESSED / f"{pdb_id}_{chainA}_{chainB}"

        for chain in ["A", "B"]:
            y_true = np.load(ex_dir / f"labels_chain{chain}_t5.npy").astype(np.int8)
            n = len(y_true)

            edge_index, _ = load_chain_graph(ex_dir, chain)
            score = baseline_surface_filtered_distance_score(n, edge_index, cfg)

            row = {
                "split": split_name,
                "example": ex_dir.name,
                "chain": chain,
                "n_res": n,
                "n_pos": int(y_true.sum()),
                "prauc": precision_recall_auc(y_true, score),
            }

            for k in k_list:
                row[f"f1@{k}"] = f1_at_k(y_true, score, k)
                row[f"p@{k}"] = precision_at_k(y_true, score, k)

            rows.append(row)

    return pd.DataFrame(rows)


<h2>Evaluate on validation set</h2>

In [13]:
df_val = eval_split("val", k_list=(10, 20, 30))
df_val.describe()


Baseline0 val:   0%|          | 0/35 [00:00<?, ?it/s]

,n_res,n_pos,prauc,f1@10,p@10,f1@20,p@20,f1@30,p@30
count,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000,70.000000
mean,250.585714,64.700000,0.353190,0.113801,0.417143,0.188836,0.402143,0.233277,0.380952
std,95.160504,45.916228,0.194813,0.074881,0.301192,0.093788,0.252155,0.111353,0.240810
min,34.000000,6.000000,0.074443,0.000000,0.000000,0.000000,0.000000,0.025316,0.033333
25%,207.500000,25.000000,0.184580,0.065054,0.125000,0.137429,0.150000,0.153784,0.166667
50%,218.000000,51.500000,0.338071,0.111735,0.400000,0.181818,0.400000,0.225000,0.333333
75%,281.000000,89.750000,0.469003,0.153846,0.600000,0.225856,0.550000,0.307692,0.533333
max,557.000000,177.000000,0.798327,0.333333,1.000000,0.600000,0.950000,0.680000,0.933333


<h2> Report baseline summary</h2>

In [14]:
summary = df_val[["prauc", "f1@10", "f1@20", "f1@30", "p@10", "p@20", "p@30"]].mean()
summary


prauc    0.353190
f1@10    0.113801
f1@20    0.188836
f1@30    0.233277
p@10     0.417143
p@20     0.402143
p@30     0.380952
dtype: float64